In [1]:
import os
import csv
import cv2
import tempfile
import pandas as pd
from PIL import Image
from feat import Detector

# ----------------------------
# Setup Detector (Py-Feat)
# ----------------------------
detector = Detector(
    face_model="retinaface",
    landmark_model="mobilefacenet",
    au_model="xgb",
    emotion_model="resmasknet"
)

# Define Action Units (AUs)
au_base = [
    "AU01", "AU02", "AU04", "AU05", "AU06", "AU07", "AU09", "AU10", "AU11", "AU12",
    "AU14", "AU15", "AU17", "AU20", "AU23", "AU24", "AU25", "AU26", "AU28", "AU43"
]

# ----------------------------
# Mapping: MediaPipe 468 -> dlib-style 68 face landmarks
# ----------------------------
DLIB_68_IDXS = [
    127, 234, 93, 132, 58, 172, 150, 176, 152, 400, 379, 378, 365, 397, 288, 361, 323, 454,
    70, 63, 105, 66, 107, 336, 296, 334, 293, 300, 383, 353, 372, 340, 346, 280, 352,
    33, 160, 158, 133, 153, 144,
    362, 385, 387, 263, 373, 380,
    168, 6, 197, 195, 5, 4, 75, 97, 2, 326, 305, 294, 278, 331, 279, 429, 358,
    61, 40, 37, 0, 267, 270, 409, 291, 375, 321, 405, 314, 17, 84, 181, 91, 146,
    78, 81, 13, 311, 402, 14, 178, 87, 317
]

# Pose landmarks (upper body only: shoulders, elbows, wrists, hands)
POSE_IDX = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]

# ----------------------------
# Extract FAUs from frame
# ----------------------------
def extract_aus_from_frame(frame):
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmpfile:
        tmp_path = tmpfile.name
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        Image.fromarray(rgb_frame).save(tmp_path)

    feat_result = detector.detect_image(tmp_path)
    os.remove(tmp_path)

    if feat_result.empty:
        return {au: None for au in au_base}
    else:
        aus = feat_result.aus.iloc[0].to_dict()
        return aus

# ----------------------------
# Extract 68 Face + Upper Body Landmarks
# ----------------------------
def extract_landmarks_from_frame(frame, holistic):
    import mediapipe as mp
    results = holistic.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    landmarks = {"face": {}, "pose": {}}

    # Face landmarks (68 points only)
    if results.face_landmarks:
        for idx, lm_idx in enumerate(DLIB_68_IDXS):
            lm = results.face_landmarks.landmark[lm_idx]
            landmarks["face"][f"F{idx+1}_x"] = lm.x
            landmarks["face"][f"F{idx+1}_y"] = lm.y
            landmarks["face"][f"F{idx+1}_z"] = lm.z

    # Upper-body pose landmarks
    if results.pose_landmarks:
        for idx, lm_idx in enumerate(POSE_IDX):
            lm = results.pose_landmarks.landmark[lm_idx]
            landmarks["pose"][f"B{idx+1}_x"] = lm.x
            landmarks["pose"][f"B{idx+1}_y"] = lm.y
            landmarks["pose"][f"B{idx+1}_z"] = lm.z

    return landmarks

# ----------------------------
# Extract Features (Start, Mid, End)
# ----------------------------
def extract_features_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count < 3:
        cap.release()
        return None

    # Pick 3 frames: start, middle, end
    frame_indices = [0, frame_count // 2, frame_count - 1]
    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)

    cap.release()

    import mediapipe as mp
    mp_holistic = mp.solutions.holistic
    with mp_holistic.Holistic(
        static_image_mode=True,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=True
    ) as holistic:

        fau_results = {au: [] for au in au_base}
        lm_results = {}

        for frame in frames:
            # Extract FAUs
            aus = extract_aus_from_frame(frame)
            for au in au_base:
                fau_results[au].append(aus.get(au, None))

            # Extract Landmarks
            landmarks = extract_landmarks_from_frame(frame, holistic)

            # Store face landmarks
            for k, v in landmarks["face"].items():
                if k not in lm_results:
                    lm_results[k] = []
                lm_results[k].append(v)

            # Store body landmarks
            for k, v in landmarks["pose"].items():
                if k not in lm_results:
                    lm_results[k] = []
                lm_results[k].append(v)

    return fau_results, lm_results

# ----------------------------
# Process Dataset Recursively
# ----------------------------
def process_dataset(root_path, labels_csv, output_csv):
    # Load labels
    labels = {}
    with open(labels_csv, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            clip_id = row["ClipID"]
            labels[clip_id] = {
                "Boredom": row["Boredom"],
                "Engagement": row["Engagement"],
                "Confusion": row["Confusion"],
                "Frustration": row["Frustration"]
            }

    results = []

    for dirpath, _, filenames in os.walk(root_path):
        for video_file in filenames:
            if not video_file.lower().endswith((".avi", ".mp4", ".mov", ".mkv")):
                continue

            video_path = os.path.join(dirpath, video_file)
            print(f"Processing {video_path}...")

            features = extract_features_from_video(video_path)
            if features is None:
                continue

            fau_results, lm_results = features
            row = {"ClipID": video_file}

            # Store FAUs as [start, mid, end]
            for au in fau_results:
                row[au] = fau_results[au]

            # Store Landmarks as [start, mid, end]
            for k in lm_results:
                row[k] = lm_results[k]

            # Store Labels
            if video_file in labels:
                for emotion in labels[video_file]:
                    val = int(labels[video_file][emotion])
                    row[emotion] = val

            results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)

# ----------------------------
# Run
# ----------------------------
dataset_path = "Sample"
labels_csv = "TrainLabels.csv"
output_csv = "output_features.csv"

process_dataset(dataset_path, labels_csv, output_csv)
print("✅ Feature extraction complete. Saved to", output_csv)


c:\Users\PC\anaconda3\envs\fb-tracking\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Processing Sample\110001\1100011002\1100011002.avi...


100%|██████████| 1/1 [00:01<00:00,  1.16s/it]


Processing Sample\110001\1100011003\1100011003.avi...


100%|██████████| 1/1 [00:01<00:00,  1.04s/it]


...

Processing Sample\110007\1100072084\1100072084.avi...


100%|██████████| 1/1 [00:00<00:00,  1.03it/s]


Processing Sample\110007\1100072085\1100072085.avi...


100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


✅ Feature extraction complete. Saved to output_features.csv
